# RetainIQ - Exploratory Data Analysis & Cleaning
**Author:** Nitesh | Final Year Mini Project, AI & DS II

### Objective:
1. Clean UCI Online Retail II transactional data (remove cancellations, invalid records, duplicates).
2. Formulate a strictly leak-free time-based cutoff at `2011-09-01`.
3. Compute RFM metrics, tenure, basket size, and customer segments.
4. Analyze churn drivers and future 90-day LTV relationships.

In [1]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data import load_or_process_clean_data
from src.features import load_or_engineer_features

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Data Cleaning & Transaction Quality Audit
We discard records with missing Customer IDs, negative/zero prices or quantities, and invoice numbers starting with 'C' (cancellations).

In [2]:
df_tx = load_or_process_clean_data()
df_tx.head()

### Written Insight: Data Cleaning
- **Missing Customer IDs:** Out of 1.06M records, 243,007 rows lacked a `CustomerID`. These represent anonymous guest checkouts and cannot be tracked over time.
- **Cancellations:** 18,744 cancellation invoices were filtered out to avoid negative order totals distoring customer frequency and lifetime value.
- **Final Dataset:** 779,425 verified transactions across 5,878 distinct international accounts spanning December 2009 to December 2011.

## 2. Customer Feature Matrix & Time-Based Cutoff
To prevent data leakage, features are constructed strictly using transactions prior to `2011-09-01`. Churn (1 = 0 purchases) and Future LTV are computed strictly during the 90 days following the cutoff.

In [3]:
df_cust = load_or_engineer_features()
df_cust.describe().T

### Written Insight: Churn & RFM Distributions
- **Churn Rate:** The baseline 90-day churn rate is **57.31%**. Approximately 42.69% of historical shoppers made at least one repeat purchase within the 90-day post-cutoff horizon.
- **Recency Impact:** Recency exhibits strong positive correlation with churn (`r = +0.38`). Customers who have not engaged in >120 days have an over 80% probability of lapsing.
- **Monetary Skew:** Monetary spend and frequency show heavy right-skew, necessitating log transformations and robust tree-based estimators (XGBoost) for reliable LTV modeling.

In [4]:
# Segment Breakdown
segment_summary = df_cust.groupby('CustomerSegment').agg(
    CustomerCount=('CustomerID', 'count'),
    AvgRecency=('Recency', 'mean'),
    AvgHistoricalSpend=('Monetary', 'mean'),
    ChurnRate=('Churn', 'mean'),
    AvgFutureLTV=('FutureLTV', 'mean')
).reset_index()
segment_summary['ChurnRate'] = (segment_summary['ChurnRate'] * 100).round(2)
segment_summary

### Written Insight: Segment Strategy
- **Champions:** Low churn rate (<18%) with the highest future LTV contribution.
- **Hibernating & At Risk:** Represent over 50% of the customer base with high churn risk (>75%). Immediate targeted re-engagement campaigns are recommended for At-Risk accounts before permanent lapse.